# Setup
## Mount Google Drive
Mounts Drive so the synthetic CSV dataset can be read from `MyDrive/synthetic dataset/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Install & Load NLP Tools
Installs spaCy/rapidfuzz, loads the transformer NER model, and reads the FIR narratives and persons table.

In [2]:
import pandas as pd
import spacy
import re

!pip install spacy rapidfuzz pandas -q
!python -m spacy download en_core_web_trf -q

nlp = spacy.load("en_core_web_trf")

firs = pd.read_csv("/content/drive/MyDrive/synthetic dataset/fir_text.csv")
persons = pd.read_csv("/content/drive/MyDrive/synthetic dataset/persons.csv")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 3.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Entity Extraction from FIR Text
## Regex-Based Structured Extraction
Parses each FIR narrative with a regex tuned to its fixed template, pulling out complainant, accused, Aadhar, phone, city, state, and date directly.

In [3]:
FIR_PATTERN = re.compile(
    r"FIR No (?P<fir_id>\d+): On (?P<date>\d{4}-\d{2}-\d{2}) "
    r"(?P<complainant>[A-Za-z ]+?) \(Aadhar (?P<aadhar>\d+)\) "
    r"reported a theft at (?P<city>[A-Za-z ]+), (?P<state>[A-Za-z ]+)\. "
    r"Suspect (?P<accused>[A-Za-z ]+?) \(Phone (?P<phone>\+91-\d+)\) "
    r"was seen near the scene\."
)

def extract_structured_fir(narrative):
    m = FIR_PATTERN.search(narrative)
    return m.groupdict() if m else None

firs["parsed"] = firs["narrative"].apply(extract_structured_fir)
print(firs["parsed"].isna().sum(), "narratives failed to parse")


0 narratives failed to parse


## NER Fallback Extraction
Runs spaCy NER over the same narratives to catch entities the regex might miss, or to generalize to messier free text later.

In [4]:
def extract_entities_ner(text, doc_id):
    doc = nlp(text)
    return [{"doc_id": doc_id, "text": ent.text, "label": ent.label_}
            for ent in doc.ents if ent.label_ in ("PERSON", "GPE", "ORG", "DATE", "MONEY")]

all_ner_entities = []
for _, row in firs.iterrows():
    all_ner_entities.extend(extract_entities_ner(row["narrative"], row["fir_id"]))

ner_df = pd.DataFrame(all_ner_entities)


## Build Clean FIR Entities Table
Joins FIR records back to `persons` on `complainant_id`, avoiding name-based fuzzy matching since we already have clean IDs.

In [5]:
fir_entities = firs[["fir_id", "complainant_id", "accused_id", "location_id", "date"]].copy()
fir_entities = fir_entities.merge(persons[["person_id", "name", "ring_id"]],
                                    left_on="complainant_id", right_on="person_id", suffixes=("", "_complainant"))


## FIR Relationship Edges
Turns each FIR's complainant/accused pair into a `fir_complaint` edge for the graph.

In [6]:
fir_edges = firs[["complainant_id", "accused_id", "fir_id", "date"]].rename(
    columns={"complainant_id": "source", "accused_id": "target"})
fir_edges["relation_type"] = "fir_complaint"


# Graph Construction
## Build the Unified Edge List
Normalizes calls, transactions, and FIR complaints into one common `(source, target, relation_type, weight, timestamp)` table.

In [8]:
import pandas as pd
import networkx as nx

calls = pd.read_csv("/content/drive/MyDrive/synthetic dataset/calls.csv")
txns = pd.read_csv("/content/drive/MyDrive/synthetic dataset/transactions.csv")
firs = pd.read_csv("/content/drive/MyDrive/synthetic dataset/fir_text.csv")
persons = pd.read_csv("/content/drive/MyDrive/synthetic dataset/persons.csv")

call_edges = calls.rename(columns={"caller_id": "source", "callee_id": "target",
                                     "start_time": "timestamp"})
call_edges["relation_type"] = "call"
call_edges["weight"] = call_edges["duration_sec"]

txn_edges = txns.rename(columns={"sender_id": "source", "receiver_id": "target",
                                   "txn_time": "timestamp"})
txn_edges["relation_type"] = "transaction"
txn_edges["weight"] = txn_edges["amount_inr"]

fir_edges = firs.rename(columns={"complainant_id": "source", "accused_id": "target",
                                   "date": "timestamp"})
fir_edges["relation_type"] = "fir_complaint"
fir_edges["weight"] = 1  # single event, no natural magnitude

edge_cols = ["source", "target", "relation_type", "weight", "timestamp"]
all_edges = pd.concat([call_edges[edge_cols], txn_edges[edge_cols], fir_edges[edge_cols]],
                       ignore_index=True)


## Build the Initial Multigraph
Loads the unified edges into a NetworkX `MultiGraph` and attaches person names as node attributes.

In [16]:
G = nx.from_pandas_edgelist(
    all_edges,
    source="source",
    target="target",
    edge_attr=["relation_type", "weight", "timestamp"],
    create_using=nx.MultiGraph()
)

for index, row in persons.iterrows():
    if row['person_id'] in G:
        G.nodes[row['person_id']]['name'] = row['name']

print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")


Number of nodes: 500
Number of edges: 3800


## Diagnostic: Inspect Edge Attributes
Sanity check — prints a few edges to confirm which attribute keys actually landed on them.

In [12]:
for u, v, data in list(G.edges(data=True))[:5]:
    print(u, v, data)


418 197 {'relation_type': 'call', 'weight': 2075.0, 'timestamp': '2026-08-16T18:18:39'}
418 375 {'relation_type': 'call', 'weight': 1591.0, 'timestamp': '2026-07-30T20:27:25'}
418 315 {'relation_type': 'call', 'weight': 3316.0, 'timestamp': '2026-08-12T01:56:48'}
418 75 {'relation_type': 'call', 'weight': 3434.0, 'timestamp': '2026-08-18T04:47:47'}
418 380 {'relation_type': 'call', 'weight': 2809.0, 'timestamp': '2026-08-22T23:49:16'}


## Fix: Re-Add Edges with the Correct `relation` Key
The diagnostic above showed edges were missing the `relation` attribute; this re-adds them with the key name the rest of the pipeline expects.

In [13]:
for _, e in all_edges.iterrows():
    G.add_edge(e["source"], e["target"],
               relation=e["relation_type"],
               weight=e["weight"],
               timestamp=e["timestamp"])


## Build the Weighted Simple Graph (`G_simple`)
Collapses the multigraph into a single weighted `Graph`: edge weight = interaction count, `relations` = which channels (call/transaction/FIR) connect that pair.

In [18]:
G_simple = nx.Graph()
G_simple.add_nodes_from(G.nodes(data=True))

for index, row in persons.iterrows():
    if row['person_id'] in G_simple:
        G_simple.nodes[row['person_id']]['name'] = row['name']

edge_weights = {}
for u, v, data in G.edges(data=True):
    key = tuple(sorted((u, v)))
    edge_weights.setdefault(key, {"count": 0, "relations": set()})
    edge_weights[key]["count"] += 1
    edge_weights[key]["relations"].add(data.get("relation", "unknown"))

for (u, v), info in edge_weights.items():
    G_simple.add_edge(u, v, weight=info["count"], relations=list(info["relations"]))

print(f"G_simple: {G_simple.number_of_nodes()} nodes, {G_simple.number_of_edges()} edges")


G_simple: 500 nodes, 3757 edges


# Network Analytics
## Key Player Identification
Computes degree, betweenness, and PageRank centrality, and prints the top brokers by betweenness.

In [19]:
degree_cent = nx.degree_centrality(G_simple)
betweenness = nx.betweenness_centrality(G_simple, weight="weight")
pagerank = nx.pagerank(G_simple, weight="weight")

top_brokers = sorted(betweenness.items(), key=lambda x: -x[1])[:10]
for pid, score in top_brokers:
    name = G_simple.nodes[pid]["name"]
    print(f"{name} (id={pid}): betweenness={score:.4f}")


Fitan Samra (id=168): betweenness=0.0091
Megha Ravi (id=461): betweenness=0.0084
Yagnesh Shan (id=91): betweenness=0.0083
Ojas Kuruvilla (id=445): betweenness=0.0079
Pratyush Raghavan (id=391): betweenness=0.0078
Siddharth Doctor (id=442): betweenness=0.0077
Advay Contractor (id=13): betweenness=0.0077
Ekani Iyer (id=258): betweenness=0.0077
Varenya Kibe (id=233): betweenness=0.0076
Deepa Chokshi (id=62): betweenness=0.0076


## Community Detection
Runs Louvain community detection and cross-checks each detected community against the dataset's ground-truth `ring_id`.

In [22]:
from networkx.algorithms.community import louvain_communities
import pandas as pd
from collections import Counter

# fill ring_id on every node so lookups below never KeyError
for index, row in persons.iterrows():
    if row['person_id'] in G_simple:
        G_simple.nodes[row['person_id']]['ring_id'] = row['ring_id']

communities = louvain_communities(G_simple, weight="weight", seed=42)
print(f"Found {len(communities)} communities")

for i, comm in enumerate(communities):
    ring_ids_in_comm = [G_simple.nodes[n]["ring_id"] for n in comm if not pd.isna(G_simple.nodes[n]["ring_id"])]
    if ring_ids_in_comm:
        print(f"Community {i} (size {len(comm)}): dominant ring = {Counter(ring_ids_in_comm).most_common(1)}")


Found 11 communities
Community 0 (size 32): dominant ring = [(0.0, 5)]
Community 1 (size 69): dominant ring = [(2.0, 7)]
Community 2 (size 90): dominant ring = [(1.0, 9)]
Community 3 (size 47): dominant ring = [(2.0, 5)]
Community 4 (size 25): dominant ring = [(4.0, 3)]
Community 5 (size 30): dominant ring = [(1.0, 3)]
Community 6 (size 72): dominant ring = [(3.0, 10)]
Community 7 (size 25): dominant ring = [(1.0, 2)]
Community 8 (size 33): dominant ring = [(0.0, 3)]
Community 9 (size 23): dominant ring = [(0.0, 4)]
Community 10 (size 54): dominant ring = [(4.0, 6)]


# Export
## Save the Graph
Pickles `G_simple` to disk for reuse in the dashboard or further analysis.

In [23]:
import pickle
with open("criminal_network.gpickle", "wb") as f:
    pickle.dump(G_simple, f)
